SETUP THE NWB DATA

In [1]:
import sys
sys.path.insert(0, '/code/src')

import glob
import os
import numpy as np
import pandas as pd
import seaborn as sns
import pynwb
from matplotlib import pyplot as plt
from matplotlib.patches import Patch
from datetime import datetime
from scipy.stats import linregress

sns.set_theme(context='talk', style='ticks', palette='colorblind')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['axes.titlesize'] = 'medium'
plt.rcParams['axes.titlelocation'] = 'left'

sns.set_context('talk')
pd.set_option('display.max_columns', None)

DATA_ROOT = '/data/dynamicrouting_datacube'

SESSION_NUMBER = 1        # <-- change this to switch session

# every session available on disk, sorted so the numbering is stable
session_paths = sorted(glob.glob('%s/*/*.nwb.zarr' % DATA_ROOT))
session_ids = [os.path.basename(p).replace('.nwb.zarr', '') for p in session_paths]
print('%d sessions available' % len(session_ids))
for position, available_id in enumerate(session_ids):
    print('  %2d  %s' % (position, available_id))

session_id = session_ids[SESSION_NUMBER]
nwb_path = session_paths[SESSION_NUMBER]

# cache: re-running this cell for the same session does not reload the file
if 'session_cache' not in globals():
    session_cache = {}
if session_id not in session_cache:
    print('\nloading %s ...' % session_id)
    session_cache[session_id] = pynwb.read_nwb(nwb_path)
else:
    print('\nusing cached %s' % session_id)

session = session_cache[session_id]
trials = session.trials[:]
print('session %d: %s, %d trials' % (SESSION_NUMBER, session_id, len(trials)))

12 sessions available
   0  662892_2023-08-24
   1  664851_2023-11-16
   2  667252_2023-09-28
   3  708016_2024-04-29
   4  712815_2024-05-22
   5  713655_2024-08-09
   6  714748_2024-06-24
   7  715710_2024-07-16
   8  741137_2024-10-10
   9  742903_2024-10-23
  10  743199_2024-12-05
  11  759434_2025-02-04

loading 664851_2023-11-16 ...


/opt/conda/lib/python3.12/site-packages/hdmf_zarr/backend.py:1699: UserWarning: Inferred dtype from zarr type. Dataset missing zarr_dtype: data   <zarr.core.Array '/processing/behavior/facemap_front_camera/data' (400638, 500) float32 read-only>
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/hdmf_zarr/backend.py:1699: UserWarning: Inferred dtype from zarr type. Dataset missing zarr_dtype: data   <zarr.core.Array '/processing/behavior/facemap_side_camera/data' (400680, 500) float32 read-only>
  warnings.warn(


session 1: 664851_2023-11-16, 527 trials


SETUP THE METADATA

In [2]:
from aind_data_access_api.document_db import MetadataDbClient

API_GATEWAY_HOST = 'api.allenneuraldynamics.org'
DATABASE = 'metadata_index'
COLLECTION = 'data_assets'

docdb_api_client = MetadataDbClient(host=API_GATEWAY_HOST, version='v2',
                                    database=DATABASE, collection=COLLECTION)

aggregate = [
    {'$match': {
        'data_description.project_name': 'Dynamic Routing',
        'data_description.data_level': 'derived',
        'processing.data_processes': {'$elemMatch': {
            'process_type': 'File format conversion',
            'start_date_time': {'$regex': '^2026-08-04'}}}}},
    {'$project': {
        'name': 1,
        'subject_id': '$data_description.subject_id',
        'genotype': '$subject.subject_details.genotype',
        'date_of_birth': '$subject.subject_details.date_of_birth',
        'sex': '$subject.subject_details.sex',
        'session_start_time': '$acquisition.acquisition_start_time',
        'session_end_time': '$acquisition.acquisition_end_time',
        'stimulus_epochs': '$acquisition.stimulus_epochs',
        'project_name': '$data_description.project_name',
        'modality': '$data_description.modalities.name',
        'targeted_structure': ('$acquisition.data_streams.configurations.probes'
                               '.primary_targeted_structure.name')}},
]

records = docdb_api_client.aggregate_docdb_records(pipeline=aggregate)
for r in records:
    dr = next((e for e in r.get('stimulus_epochs', [])
               if e.get('stimulus_name') == 'DynamicRouting1'), None)
    r['dr_performance'] = dr['performance_metrics'] if dr else None

metadata_df = pd.DataFrame(records)
metadata_df['trials_total'] = metadata_df['dr_performance'].apply(
    lambda x: x['trials_total'] if x else None)
metadata_df['trials_rewarded'] = metadata_df['dr_performance'].apply(
    lambda x: x['trials_rewarded'] if x else None)

# independent cross-check: these counts come from the metadata index, not the NWB file
this_session_row = metadata_df[metadata_df.name.str.contains(session_id, na=False)]
print('Metadata trials_total: %s, NWB trials: %d'
      % (this_session_row.trials_total.values, len(trials)))
print('Metadata trials_rewarded: %s, NWB is_rewarded: %d'
      % (this_session_row.trials_rewarded.values, trials.is_rewarded.sum()))
metadata_df.head(3)

Metadata trials_total: [527], NWB trials: 527
Metadata trials_rewarded: [139], NWB is_rewarded: 145


,_id,name,subject_id,genotype,date_of_birth,sex,session_start_time,session_end_time,stimulus_epochs,project_name,modality,targeted_structure,dr_performance,trials_total,trials_rewarded
0,c5bc8e0e-81ee-4e52-a7b6-90089b0dfc5f,ecephys_742903_2024-10-23_14-12-23_nwb_2026-08...,742903,Vip-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,2024-05-16,Female,2024-10-23 14:12:23-07:00,2024-10-23 16:15:54.652646-07:00,"[{'object_type': 'Stimulus epoch', 'stimulus_s...",Dynamic Routing,"[Extracellular electrophysiology, Behavior, Be...","[[[Primary somatosensory area], [Caudoputamen]...","{'object_type': 'Performance metrics', 'output...",489,124
1,7f603b00-8b9f-4a66-8b9d-1d9ed27b40d4,ecephys_743199_2024-12-05_12-42-34_nwb_2026-08...,743199,VGAT-ChR2-YFP/wt,2024-05-18,Female,2024-12-05 12:42:34-08:00,2024-12-05 14:39:42.445214-08:00,"[{'object_type': 'Stimulus epoch', 'stimulus_s...",Dynamic Routing,"[Extracellular electrophysiology, Behavior, Be...","[[[Periaqueductal gray], [Red nucleus], [Red n...","{'object_type': 'Performance metrics', 'output...",490,132
2,66dc0f20-45dc-4a65-ac3a-0a04ac0e1df4,ecephys_662892_2023-08-24_14-28-28_nwb_2026-08...,662892,Sst-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,2022-12-24,Female,2023-08-24 14:28:28-07:00,2023-08-24 16:28:14.588745-07:00,"[{'object_type': 'Stimulus epoch', 'stimulus_s...",Dynamic Routing,"[Extracellular electrophysiology, Behavior, Be...","[[[Piriform area], [root]]]","{'object_type': 'Performance metrics', 'output...",476,123


UNITS QUALITY CONTROL

In [ ]:
units_table = session.units[:]
print('This session has %d units before quality control' % len(units_table))

# QC thresholds
MAX_ISI_VIOLATIONS_RATIO = 0.5    # lower is better
MAX_AMPLITUDE_CUTOFF = 0.1        # lower is better
MIN_PRESENCE_RATIO = 0.95         # higher is better

good_units = units_table[
    (units_table.isi_violations_ratio < MAX_ISI_VIOLATIONS_RATIO) &
    (units_table.amplitude_cutoff < MAX_AMPLITUDE_CUTOFF) &
    (units_table.presence_ratio > MIN_PRESENCE_RATIO)
]

this_structure_units_table = good_units[good_units.structure == 'MOs']
print('%d good units in MOs' % len(this_structure_units_table))

TRIAL STRUCTURE

In [ ]:
# Trial structure
print('Trial structure: go %d + nogo %d + catch %d = %d, total trials %d'
      % (trials.is_go.sum(), trials.is_nogo.sum(), trials.is_catch.sum(),
         trials.is_go.sum() + trials.is_nogo.sum() + trials.is_catch.sum(),
         len(trials)))
print('hit %d + miss %d = go %d'
      % (trials.is_hit.sum(), trials.is_miss.sum(), trials.is_go.sum()))
print('false alarm %d + correct reject %d = nogo %d'
      % (trials.is_false_alarm.sum(), trials.is_correct_reject.sum(),
         trials.is_nogo.sum()))

# Responses types
n_catch_response = (trials.is_catch & trials.is_response).sum()
print('\nhit %d + false alarm %d + catch response %d = %d, is_response %d'
      % (trials.is_hit.sum(), trials.is_false_alarm.sum(), n_catch_response,
         trials.is_hit.sum() + trials.is_false_alarm.sum() + n_catch_response,
         trials.is_response.sum()))

# Response window
window_start_offset = trials.response_window_start_time - trials.stim_start_time
window_stop_offset = trials.response_window_stop_time - trials.stim_start_time
print('\nresponse window start offset: %.4f to %.4f s (median %.4f)'
      % (window_start_offset.min(), window_start_offset.max(),
         window_start_offset.median()))
print('response window stop  offset: %.4f to %.4f s (median %.4f)'
      % (window_stop_offset.min(), window_stop_offset.max(),
         window_stop_offset.median()))

# Save outcome trial structure
trials['outcome'] = np.select(
    [trials.is_hit, trials.is_miss, trials.is_false_alarm, trials.is_correct_reject],
    ['hit', 'miss', 'false alarm', 'correct reject'], default='unscored')

SETUP BEHAVIOR DATA

In [ ]:
behavior_module = session.processing['behavior']
print('behavior module contents: %s\n' % list(behavior_module.data_interfaces.keys()))

running_series = behavior_module['running_speed']
running_timestamps = running_series.timestamps[:]
running_speed = running_series.data[:]
side_camera_df = behavior_module['lp_side_camera'][:]
eye_df = behavior_module['eye_tracking'][:]
rewards_df = behavior_module['rewards'][:]

In [ ]:
TRIAL_WINDOW = 'quiescent'      # 'quiescent' = 1.5 s pre-stimulus; 'whole_trial' = start to stop
QUIESCENT_WINDOW_S = 1.5
SIDE_CAMERA_FRAME_HEIGHT = 492


def get_facial_feature(part_name, facial_features_df,
                       frame_height=SIDE_CAMERA_FRAME_HEIGHT):
    """Vertical keypoint position, masked and interpolated (workshop 1)."""
    confidence = facial_features_df['%s_likelihood' % part_name]
    temporal_norm = facial_features_df['%s_temporal_norm' % part_name]

    y = frame_height - facial_features_df['%s_y' % part_name].astype(float)
    y[(confidence < 0.99)
      | (temporal_norm > np.nanmean(temporal_norm) + 2 * np.nanstd(temporal_norm))] = np.nan

    y_centered = y - np.nanmean(y)
    y_abs_centered = np.abs(y_centered)
    y_centered[y_abs_centered > np.nanmean(y_abs_centered)
               + 2 * np.nanstd(y_abs_centered)] = np.nan
    return pd.Series(y_centered).ffill().bfill().to_numpy()


def get_trialwise_mean_and_sd(x, timestamps, start, stop):
    """Mean and standard deviation of a signal within each trial window.

    Mean gives the level, SD gives within-trial variability. Returns two arrays.
    """
    x = np.asarray(x, dtype=float)
    timestamps = np.asarray(timestamps, dtype=float)
    mean_array = np.full(len(start), np.nan)
    sd_array = np.full(len(start), np.nan)
    for trial_position, (window_start, window_stop) in enumerate(zip(start, stop)):
        in_window = (timestamps >= window_start) & (timestamps <= window_stop)
        if in_window.sum() >= 2:
            mean_array[trial_position] = np.nanmean(x[in_window])
            sd_array[trial_position] = np.nanstd(x[in_window])
    return mean_array, sd_array


# Window of analysis
if TRIAL_WINDOW == 'quiescent':
    trial_window_start = trials.stim_start_time.values - QUIESCENT_WINDOW_S
    trial_window_stop = trials.stim_start_time.values
else:
    trial_window_start = trials.start_time.values
    trial_window_stop = trials.stop_time.values

window_duration = trial_window_stop - trial_window_start

# --- continuous signals -----------------------------------------------------
running_speed_clean = pd.Series(running_speed).interpolate(limit_direction='both').to_numpy()

pupil_area = eye_df['pupil_area'].astype(float).to_numpy().copy()
pupil_area[eye_df['pupil_is_bad_frame'].to_numpy().astype(bool)] = np.nan
pupil_area = pd.Series(pupil_area).interpolate(limit_direction='both').to_numpy()

side_camera_timestamps = side_camera_df['timestamps'].values.astype(float)
eye_timestamps = eye_df['timestamps'].values.astype(float)

signal_sources = [('running_speed', running_speed_clean, running_timestamps),
                  ('pupil_area', pupil_area, eye_timestamps)]
for column_name, keypoint_name in [('ear', 'ear_base_l'), ('nose', 'nose_tip'), ('jaw', 'jaw'), ('whiskers', 'whisker_pad_l_side')]:
    signal_sources.append((column_name, get_facial_feature(keypoint_name, side_camera_df), side_camera_timestamps))

# Compute mean and SD per trial
for column_name, signal_array, timestamp_array in signal_sources:
    mean_array, sd_array = get_trialwise_mean_and_sd(signal_array, timestamp_array, trial_window_start, trial_window_stop)
    trials[column_name + '_mean'] = mean_array
    trials[column_name + '_sd'] = sd_array

BEHAVIOR_VARIABLES = [('running_speed', 'Running speed (cm/s)'),
                      ('pupil_area', 'Pupil area (px$^2$)'),
                      ('ear', 'Ear position (px)'),
                      ('nose', 'Nose position (px)'),
                      ('jaw', 'Jaw position (px)'),
                      ('whiskers', 'Whisker pad position (px)')]

summary_columns = ([c + '_mean' for c, _ in BEHAVIOR_VARIABLES] + [c + '_sd' for c, _ in BEHAVIOR_VARIABLES])
print(trials[summary_columns].describe().T[['count', 'mean', 'std', 'min', 'max']])

In [ ]:
STREAK_COLUMNS = [('consecutive_rewarded', '# consecutive rewarded trials before'), ('consecutive_unrewarded', '# consecutive unrewarded trials before')]
OUTCOME_COLORS = {'hit': 'tab:blue', 'false alarm': 'tab:red'}
RT_OUTCOMES = ['hit', 'false alarm']
STATISTICS = [('_mean', 'mean'), ('_sd', 'SD')]

behavior_streak_data = trials[~trials.is_catch & trials.outcome.isin(RT_OUTCOMES)].copy()

for behavior_column, behavior_label in BEHAVIOR_VARIABLES:
    for statistic_suffix, statistic_label in STATISTICS:
        value_column = behavior_column + statistic_suffix
        panel_data = behavior_streak_data[behavior_streak_data[value_column].notna()]

        fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)

        for col_index, (streak_column, column_label) in enumerate(STREAK_COLUMNS):
            ax = axes[col_index]
            level_order = sorted(panel_data[streak_column].unique())

            sns.boxplot(data=panel_data, x=streak_column, y=value_column,
                        hue='outcome', hue_order=RT_OUTCOMES, palette=OUTCOME_COLORS,
                        order=level_order, width=0.6, showfliers=False, linewidth=1,
                        boxprops={'alpha': 0.45},
                        legend=(col_index == 0), ax=ax)
            sns.stripplot(data=panel_data, x=streak_column, y=value_column,
                          hue='outcome', hue_order=RT_OUTCOMES, palette=OUTCOME_COLORS,
                          order=level_order, dodge=True, size=3.5, alpha=0.8,
                          linewidth=0, legend=False, ax=ax)

            ax.set_xlabel(column_label)
            ax.set_ylabel('%s, %s' % (behavior_label, statistic_label)
                          if col_index == 0 else '')
            sns.despine(ax=ax)

        legend = axes[0].get_legend()
        if legend is not None:
            legend.set_title('')
            legend.prop.set_size(9)
            legend.get_frame().set_linewidth(0)

        fig.tight_layout()
        plt.show()

In [ ]:
MIN_TRIALS_FOR_FIT = 20

behavior_streak_data = trials[~trials.is_catch & trials.outcome.isin(RT_OUTCOMES)].copy()

slope_rows = []

for behavior_column, behavior_label in BEHAVIOR_VARIABLES + [('reaction_time', None)]:
    statistic_list = STATISTICS if behavior_label is not None else [('', '')]
    for statistic_suffix, statistic_label in statistic_list:
        value_column = behavior_column + statistic_suffix
        if value_column not in behavior_streak_data:
            continue
        panel_label = ('Reaction time (s)' if behavior_label is None else '%s, %s' % (behavior_label, statistic_label))
        panel_data = behavior_streak_data[behavior_streak_data[value_column].notna()]

        fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)

        for col_index, (streak_column, column_label) in enumerate(STREAK_COLUMNS):
            ax = axes[col_index]
            level_order = sorted(panel_data[streak_column].unique())

            sns.boxplot(data=panel_data, x=streak_column, y=value_column,
                        hue='outcome', hue_order=RT_OUTCOMES, palette=OUTCOME_COLORS,
                        order=level_order, width=0.6, showfliers=False, linewidth=1,
                        boxprops={'alpha': 0.45},
                        legend=(col_index == 0), ax=ax)
            sns.stripplot(data=panel_data, x=streak_column, y=value_column,
                          hue='outcome', hue_order=RT_OUTCOMES, palette=OUTCOME_COLORS,
                          order=level_order, dodge=True, size=3.5, alpha=0.8,
                          linewidth=0, legend=False, ax=ax)

            position_by_level = {level: position for position, level in enumerate(level_order)}
            title_parts = []
            for outcome_label in RT_OUTCOMES:
                outcome_data = panel_data[panel_data.outcome == outcome_label]
                x = outcome_data[streak_column].map(
                    position_by_level).values.astype(float)
                y = outcome_data[value_column].values.astype(float)

                if len(x) < MIN_TRIALS_FOR_FIT or len(np.unique(x)) < 2:
                    title_parts.append('%s: n=%d, not fitted' % (outcome_label, len(x)))
                    continue

                result = linregress(x, y)
                x_line = np.array([x.min(), x.max()])
                ax.plot(x_line, result.intercept + result.slope * x_line, color=OUTCOME_COLORS[outcome_label], lw=2.5, zorder=5)
                title_parts.append('%s r\u00b2 %.2f' % (outcome_label, result.rvalue ** 2))

                # Normalize slopes
                slope_rows.append({
                    'variable': panel_label,
                    'streak': column_label,
                    'outcome': outcome_label,
                    'slope': result.slope,
                    'slope_std': result.slope * x.std() / y.std() if y.std() > 0 else np.nan,
                    'stderr_std': result.stderr * x.std() / y.std() if y.std() > 0 else np.nan,
                    'r2': result.rvalue ** 2,
                    'p': result.pvalue,
                    'n': len(x)})

            ax.set_xlabel(column_label)
            ax.set_ylabel(panel_label if col_index == 0 else '')
            ax.set_title('  |  '.join(title_parts), fontsize=9)
            sns.despine(ax=ax)

        legend = axes[0].get_legend()
        if legend is not None:
            legend.set_title('')
            legend.prop.set_size(9)
            legend.get_frame().set_linewidth(0)

        fig.tight_layout()
        plt.show()

slope_table = pd.DataFrame(slope_rows)
print(slope_table.sort_values('slope_std', key=np.abs, ascending=False).to_string(index=False, float_format='%.3f'))

In [ ]:
def significance_marker(p_value):
    """Conventional stars. 52 uncorrected tests here, so a single star is weak."""
    if not np.isfinite(p_value):
        return ''
    if p_value < 0.001:
        return '***'
    if p_value < 0.01:
        return '**'
    if p_value < 0.05:
        return '*'
    return 'n.s.'


# Order to show
VARIABLE_SEQUENCE = ['pupil_area', 'running_speed', 'ear', 'nose', 'jaw', 'whiskers']
label_by_column = dict(BEHAVIOR_VARIABLES)

variable_order = ['Reaction time (s)']
for statistic_label in ['mean', 'SD']:
    for behavior_column in VARIABLE_SEQUENCE:
        if behavior_column in label_by_column:
            variable_order.append('%s, %s' % (label_by_column[behavior_column], statistic_label))
variable_order = [v for v in variable_order if v in set(slope_table.variable)]

y_position = {variable: position for position, variable in enumerate(reversed(variable_order))}

fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharex=True, sharey=True)

for panel_index, (_, column_label) in enumerate(STREAK_COLUMNS):
    ax = axes[panel_index]
    panel_table = slope_table[slope_table.streak == column_label]

    for outcome_label, offset in zip(RT_OUTCOMES, [-0.2, 0.2]):
        outcome_table = panel_table[panel_table.variable.isin(y_position) & (panel_table.outcome == outcome_label)]
        positions = outcome_table.variable.map(y_position).values + offset
        ax.barh(positions, outcome_table.slope_std, height=0.35, color=OUTCOME_COLORS[outcome_label], alpha=0.7,
            label=outcome_label if panel_index == 0 else None)
        ax.errorbar(outcome_table.slope_std, positions, xerr=outcome_table.stderr_std, fmt='none', ecolor='k', lw=1, capsize=2)

        for bar_position, slope_value, stderr_value, p_value in zip(positions, outcome_table.slope_std, outcome_table.stderr_std, outcome_table.p):
            marker = significance_marker(p_value)
            if not marker:
                continue
            direction = 1 if slope_value >= 0 else -1
            label_x = slope_value + direction * (abs(stderr_value) + 0.02)
            ax.annotate(marker, (label_x, bar_position),
                        ha='left' if direction > 0 else 'right', va='center',
                        fontsize=8 if marker != 'n.s.' else 6,
                        color='k' if marker != 'n.s.' else '0.55')

    ax.axvline(0, color='k', lw=1)
    ax.set_yticks(range(len(variable_order)))
    ax.set_yticklabels(list(reversed(variable_order)))
    ax.set_xlabel('Normalized slope (SD)')
    ax.set_title(column_label, fontsize=11)
    sns.despine(ax=ax)

for ax in axes:
    x_low, x_high = ax.get_xlim()
    ax.set_xlim(x_low - 0.08, x_high + 0.08)

axes[0].legend(fontsize=9, frameon=False, loc='lower right')
fig.suptitle('Sensitivity to reward streak, %s window (\u00b1 SE; ' '* p<0.05, ** p<0.01, *** p<0.001, uncorrected)' % TRIAL_WINDOW, fontsize=10)
fig.tight_layout()
plt.show()

In [ ]:
P_THRESHOLD = 0.001

# rows where at least one outcome passes; the other is drawn hollow so the
# figure never implies only one estimate was fitted
significant_keys = set(
    slope_table.loc[slope_table.p < P_THRESHOLD, ['variable', 'streak']]
    .itertuples(index=False, name=None))
significant_table = slope_table[
    slope_table.apply(lambda r: (r.variable, r.streak) in significant_keys, axis=1)]

print('%d of %d fits at p < %.3f'
      % (int((slope_table.p < P_THRESHOLD).sum()), len(slope_table), P_THRESHOLD))

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True)

for panel_index, (_, column_label) in enumerate(STREAK_COLUMNS):
    ax = axes[panel_index]
    panel_table = significant_table[significant_table.streak == column_label]

    # keep the fixed variable order, restricted to what survived in this panel
    panel_variables = [v for v in variable_order if v in set(panel_table.variable)]
    panel_position = {variable: position
                      for position, variable in enumerate(reversed(panel_variables))}

    for outcome_label, offset in zip(RT_OUTCOMES, [-0.2, 0.2]):
        outcome_table = panel_table[panel_table.outcome == outcome_label]
        if not len(outcome_table):
            continue
        positions = outcome_table.variable.map(panel_position).values + offset
        is_significant = (outcome_table.p < P_THRESHOLD).values
        ax.barh(positions, outcome_table.slope_std, height=0.35,
                color=[OUTCOME_COLORS[outcome_label] if s else 'none'
                       for s in is_significant],
                edgecolor=OUTCOME_COLORS[outcome_label], linewidth=1.2,
                alpha=0.7, label=outcome_label if panel_index == 0 else None)
        ax.errorbar(outcome_table.slope_std, positions,
                    xerr=outcome_table.stderr_std, fmt='none',
                    ecolor='k', lw=1, capsize=2)

    ax.axvline(0, color='k', lw=1)
    ax.set_yticks(range(len(panel_variables)))
    ax.set_yticklabels(list(reversed(panel_variables)))
    ax.set_xlabel('Normalized slope (SD)')
    ax.set_title('%s (%d of %d fits)'
                 % (column_label,
                    int((slope_table[slope_table.streak == column_label].p
                         < P_THRESHOLD).sum()),
                    len(slope_table[slope_table.streak == column_label])),
                 fontsize=11)
    sns.despine(ax=ax)

for ax in axes:
    x_low, x_high = ax.get_xlim()
    ax.set_xlim(x_low - 0.08, x_high + 0.08)

axes[0].legend(fontsize=9, frameon=False, loc='lower right')
fig.suptitle('Sensitivity to reward streak, %s window (p < %.3f; '
             'hollow = same variable, other outcome, above threshold)'
             % (TRIAL_WINDOW, P_THRESHOLD), fontsize=10)
fig.tight_layout()
plt.show()